# Vascular Network Experiment Notebook

Interactive testing of the **Space Colonization** and **CCO Hybrid + NLP** backends.

No GUI required — everything runs from pure Python.

## 1. Setup & Imports

In [ ]:
import sys, os
from pathlib import Path

ROOT = str(Path(".").resolve().parent)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from test.space_colonization_runner import run_space_colonization, run_space_colonization_dual_tree
from test.cco_runner import run_cco, run_cco_dual_tree
from test.notebook_utils import (
    plot_network_2d,
    plot_network_3d,
    print_stats,
    compare_networks,
    compare_stats_table,
    network_to_dataframe,
    save_network_json,
)

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

## 2. Space Colonization
### 2a. Single-inlet (cylinder domain)

In [ ]:
sc_params = {
    "domain_type": "cylinder",
    "domain_radius": 0.005,      # 5 mm
    "domain_height": 0.010,      # 10 mm
    "domain_center": [0.0, 0.0, 0.0],

    "inlet_position": [0.0, 0.0, 0.005],
    "inlet_radius": 0.001,       # 1 mm
    "vessel_type": "arterial",

    "num_attractors": 1000,
    "attraction_distance": 0.010,
    "kill_distance": 0.002,
    "step_size": 0.002,
    "max_iterations": 500,
    "max_steps": 100,
    "branch_angle_deg": 30.0,
    "directional_bias": 0.5,
    "max_deviation_deg": 60.0,

    "encourage_bifurcation": False,
    "max_children_per_node": 2,
    "bifurcation_probability": 0.7,
    "min_attractions_for_bifurcation": 3,
    "bifurcation_angle_threshold_deg": 40.0,

    "min_radius": 0.0001,
    "taper_factor": 0.95,

    "progress": True,
    "kdtree_rebuild_tip_every": 1,
    "kdtree_rebuild_all_nodes_every": 10,
    "stall_steps_per_inlet": 10,
    "interleaving_strategy": "round_robin",

    "check_collisions": True,
    "collision_clearance": 0.0002,
    "collision_merge_distance": 0.0003,

    "seed": 42,
    "num_outlets": 50,
}

sc_net, sc_stats = run_space_colonization(sc_params)
print_stats(sc_stats, "Space Colonization -- single inlet")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, proj in zip(axes, ["xy", "xz", "yz"]):
    plot_network_2d(sc_net, projection=proj, ax=ax, title=f"SC single-inlet ({proj})")
plt.tight_layout()
plt.show()

In [ ]:
fig3d = plot_network_3d(sc_net, title="SC Single-Inlet 3D")
fig3d.show()

### 2b. Multi-inlet (blended mode)

In [ ]:
sc_multi_params = {
    **sc_params,
    "inlets": [
        {"position": [0.003, 0.0, 0.005], "radius": 0.0008},
        {"position": [-0.003, 0.0, 0.005], "radius": 0.0008},
        {"position": [0.0, 0.003, 0.005], "radius": 0.0008},
    ],
    "multi_inlet_mode": "blended",
    "multi_inlet_blend_sigma": 0.0,
    "max_inlets": 10,
    "num_attractors": 2000,
    "seed": 123,
}

sc_multi_net, sc_multi_stats = run_space_colonization(sc_multi_params)
print_stats(sc_multi_stats, "SC -- blended multi-inlet")
plot_network_2d(sc_multi_net, title="SC blended multi-inlet (xy)")
plt.show()

### 2c. Multi-inlet (partitioned_xy mode)

In [ ]:
sc_part_params = {
    **sc_multi_params,
    "multi_inlet_mode": "partitioned_xy",
    "partitioned_directional_bias": 1.0,
    "partitioned_max_deviation_deg": 30.0,
    "partitioned_cone_angle_deg": 30.0,
    "partitioned_cylinder_radius": 0.001,
    "seed": 123,
}

sc_part_net, sc_part_stats = run_space_colonization(sc_part_params)
print_stats(sc_part_stats, "SC -- partitioned_xy multi-inlet")
plot_network_2d(sc_part_net, title="SC partitioned_xy (xy)")
plt.show()

### 2d. Multi-inlet (forest mode)

In [ ]:
sc_forest_params = {
    **sc_multi_params,
    "multi_inlet_mode": "forest",
    "seed": 123,
}

sc_forest_net, sc_forest_stats = run_space_colonization(sc_forest_params)
print_stats(sc_forest_stats, "SC -- forest multi-inlet")
plot_network_2d(sc_forest_net, title="SC forest mode (xy)")
plt.show()

### 2e. Parameter sweep -- num_attractors

In [ ]:
sweep_results = {}
for n_att in [500, 1000, 2000, 4000]:
    p = {**sc_params, "num_attractors": n_att, "seed": 42}
    net, stats = run_space_colonization(p)
    sweep_results[f"attractors={n_att}"] = (net, stats)

compare_networks(sweep_results, projection="xy")

In [ ]:
compare_stats_table({k: v[1] for k, v in sweep_results.items()})

### 2f. Dual arterial-venous tree (Space Colonization)

In [ ]:
sc_dual_params = {
    "domain_type": "cylinder",
    "domain_radius": 0.005,
    "domain_height": 0.010,
    "num_attractors": 1500,
    "step_size": 0.002,
    "max_iterations": 500,
    "arterial_outlets": 30,
    "venous_outlets": 30,
    "arterial_radius": 0.001,
    "venous_radius": 0.001,
    "create_anastomoses": False,
    "seed": 42,
    "progress": True,
}

sc_dual_net, sc_dual_stats = run_space_colonization_dual_tree(sc_dual_params)
print_stats(sc_dual_stats, "SC Dual Tree")
plot_network_2d(sc_dual_net, title="SC Dual Tree (xy)", color_by="vessel_type")
plt.show()

## 3. CCO Hybrid + NLP
### 3a. Single tree (grid search)

In [ ]:
cco_params = {
    "domain_type": "cylinder",
    "domain_radius": 0.005,
    "domain_height": 0.010,

    "inlet_position": [0.0, 0.0, 0.005],
    "inlet_radius": 0.001,
    "vessel_type": "arterial",

    "murray_exponent": 3.0,

    "cost_length_weight": 1.0,
    "cost_radius_weight": 1.0,
    "boundary_penalty_weight": 10.0,

    "optimization_grid_resolution": 10,
    "candidate_edges_k": 50,
    "use_nlp_optimization": False,

    "use_partial_binding": True,
    "use_collision_triage": True,

    "collision_check_enabled": True,
    "collision_clearance": 0.0001,

    "candidate_search_radius": 0.05,
    "boundary_penalty_threshold": 0.002,
    "initial_radius_taper": 0.8,
    "fallback_murray_split_ratio": 0.8,
    "outlet_end_radius_taper": 0.9,
    "single_child_taper": 0.9,

    "max_consecutive_failures": 50,
    "default_inlet_radius": 0.002,

    "enable_trifurcation": False,
    "trifurcation_cost_threshold": 0.8,

    "seed": 42,
    "num_outlets": 50,
    "min_segment_length": 0.0005,
    "max_segment_length": 0.020,
    "min_radius": 0.0001,
    "min_terminal_separation": 0.0005,
    "check_collisions": True,
}

cco_net, cco_stats = run_cco(cco_params)
print_stats(cco_stats, "CCO Hybrid -- grid search")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, proj in zip(axes, ["xy", "xz", "yz"]):
    plot_network_2d(cco_net, projection=proj, ax=ax, title=f"CCO grid ({proj})")
plt.tight_layout()
plt.show()

In [ ]:
fig3d = plot_network_3d(cco_net, title="CCO Grid Search 3D")
fig3d.show()

### 3b. NLP optimization (SLSQP)

In [ ]:
cco_nlp_params = {
    **cco_params,
    "use_nlp_optimization": True,
    "nlp_solver": "SLSQP",
    "nlp_tolerance": 1e-6,
    "max_nlp_iterations": 100,
    "nlp_use_grid_initial_guess": True,
    "nlp_grid_resolution_for_guess": 5,
}

cco_nlp_net, cco_nlp_stats = run_cco(cco_nlp_params)
print_stats(cco_nlp_stats, "CCO Hybrid -- NLP (SLSQP)")
plot_network_2d(cco_nlp_net, title="CCO NLP-SLSQP (xy)")
plt.show()

### 3c. NLP solver comparison

In [ ]:
solver_results = {}
for solver in ["SLSQP", "trust-constr", "L-BFGS-B"]:
    p = {**cco_params, "use_nlp_optimization": True, "nlp_solver": solver, "seed": 42}
    net, stats = run_cco(p)
    solver_results[f"NLP-{solver}"] = (net, stats)

solver_results["Grid (baseline)"] = (cco_net, cco_stats)
compare_networks(solver_results, projection="xy")

In [ ]:
compare_stats_table({k: v[1] for k, v in solver_results.items()})

### 3d. Trifurcation (3-way splits)

In [ ]:
cco_tri_params = {
    **cco_params,
    "enable_trifurcation": True,
    "trifurcation_cost_threshold": 0.8,
    "seed": 42,
}

cco_tri_net, cco_tri_stats = run_cco(cco_tri_params)
print_stats(cco_tri_stats, "CCO Hybrid -- trifurcation")
plot_network_2d(cco_tri_net, title="CCO trifurcation (xy)")
plt.show()

### 3e. Dual arterial-venous tree (CCO)

In [ ]:
cco_dual_params = {
    "domain_type": "cylinder",
    "domain_radius": 0.005,
    "domain_height": 0.010,
    "murray_exponent": 3.0,
    "optimization_grid_resolution": 10,
    "candidate_edges_k": 50,
    "arterial_outlets": 30,
    "venous_outlets": 30,
    "arterial_radius": 0.001,
    "venous_radius": 0.001,
    "create_anastomoses": False,
    "num_anastomoses": 0,
    "min_terminal_separation_same_type": 0.0005,
    "min_terminal_separation_cross_type": 0.001,
    "encourage_av_proximity": True,
    "anastomosis_max_length": 0.015,
    "seed": 42,
}

cco_dual_net, cco_dual_stats = run_cco_dual_tree(cco_dual_params)
print_stats(cco_dual_stats, "CCO Dual Tree")
plot_network_2d(cco_dual_net, title="CCO Dual Tree (xy)", color_by="vessel_type")
plt.show()

## 4. Head-to-Head Comparison

In [ ]:
common = {
    "domain_type": "cylinder",
    "domain_radius": 0.005,
    "domain_height": 0.010,
    "inlet_position": [0.0, 0.0, 0.005],
    "inlet_radius": 0.001,
    "num_outlets": 50,
    "seed": 42,
}

sc_h2h_net, sc_h2h_stats = run_space_colonization({**common, "num_attractors": 1500})
cco_h2h_net, cco_h2h_stats = run_cco(common)

compare_networks({
    "Space Colonization": (sc_h2h_net, sc_h2h_stats),
    "CCO Hybrid": (cco_h2h_net, cco_h2h_stats),
}, projection="xy")

In [ ]:
compare_stats_table({
    "Space Colonization": sc_h2h_stats,
    "CCO Hybrid": cco_h2h_stats,
})

## 5. Segment-Level Analysis

In [ ]:
df = network_to_dataframe(sc_net)
df.head(10)

In [ ]:
df["length_mm"] = df["length_m"] * 1000
df["mean_radius_um"] = df["mean_radius_m"] * 1e6

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df["length_mm"], bins=30, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Segment length (mm)")
axes[0].set_ylabel("Count")
axes[0].set_title("Segment length distribution")

axes[1].hist(df["mean_radius_um"], bins=30, color="indianred", edgecolor="white")
axes[1].set_xlabel("Mean radius (um)")
axes[1].set_ylabel("Count")
axes[1].set_title("Vessel radius distribution")

plt.tight_layout()
plt.show()

## 6. Export

In [ ]:
save_network_json(sc_net, "sc_network.json")
save_network_json(cco_net, "cco_network.json")
print("Networks saved to sc_network.json and cco_network.json")

## 7. Sandbox

Use cells below for your own experiments.

In [ ]:
# Your experiment here
